In [5]:
# Librairies nécessaires
import os
import pickle
import fasttext
import re
from gensim.models import Word2Vec, KeyedVectors
from gensim.models.fasttext import load_facebook_model
from top2vec import Top2Vec


In [6]:

project_path = r"D:\Datascientest\Projet - TrustPilot\Notebook\data"
file_name = 'dataset_final.pkl'

# chargement des données
with open(os.path.join(project_path, file_name), 'rb') as f:
    df = pickle.load(f)

In [7]:
# Préparation des documents pour les modèles
docs = df["clean_comment"].tolist()

In [8]:
import re

# Votre liste de stopwords
custom_stopwords = {
    'montre', 'montres', 'boucles', 'oreilles', 'oreille', 'paire', 'paires',
    'lunettes', 'bracelet', 'bracelets', 'collier', 'iphone', 'téléphone',
    'pendentif', 'robot', 'robots', 'aspirateur', 'baskets', 'basket',
    'chaussures', 'sandales', 'plantes', 'plante', 'arbres', 'arbre',
    'bulbes', 'willemse', 'sommiers', 'jardin', 'bague', 'lampe', 'lampes',
    'abat-jour', 'lampadaire', 'parfum', 'shampoing', 'shampooing',
    'cheveux', 'masque', 'masques', 'crème', 'élastiques', 'manteau',
    'bougie', 'cadre', 'écouteurs', 'vélo'
}

def preprocess(doc, stopwords=custom_stopwords):
    """
    Preprocess a document:
    - Lowercase
    - Remove punctuation/numbers
    - Remove custom stopwords
    """
    doc = doc.lower()
    doc = re.sub(r'[^a-zà-ÿ\s]', '', doc)  # keeps accented letters
    words = [w for w in doc.split() if w not in stopwords]
    return ' '.join(words)

# Appliquer à tous les documents
docs_clean = [preprocess(doc) for doc in docs]

In [ ]:
# Création du modèle Top2Vec
model = Top2Vec(
    documents=docs_clean,
    embedding_model='distiluse-base-multilingual-cased',  # or 'doc2vec' (smaller, faster)
    speed='learn',
    workers=4
)

2025-11-21 14:19:26,873 - top2vec - INFO - Pre-processing documents for training
c:\ProgramData\anaconda3\envs\nlp_tools\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2025-11-21 14:19:30,177 - top2vec - INFO - Downloading distiluse-base-multilingual-cased model
2025-11-21 14:19:33,524 - top2vec - INFO - Creating joint document/word embedding


In [ ]:
# Exploration des topics
print("Number of topics:", model.get_num_topics())

In [ ]:
topic = model.get_topics(3)
topic


In [ ]:
import pandas as pd
topic_sizes, topic_nums = model.get_topic_sizes()
topic_df = pd.DataFrame({
    "Topic": topic_nums,
    "Number of Reviews": topic_sizes
}).sort_values(by="Number of Reviews", ascending=False)

print(topic_df.head(10))

In [ ]:
# Visualisation des topics
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.bar(topic_df['Topic'], topic_df['Number of Reviews'])
plt.xlabel('Topic Number')
plt.ylabel('Number of Reviews')
plt.title('Number of Reviews per Topic')
plt.show()

In [ ]:
plt.figure(figsize=(8,8))
plt.pie(topic_df['Number of Reviews'], labels=topic_df['Topic'])
plt.title('Proportion of Reviews per Topic')
plt.show()

In [ ]:
# Visual evaluation
# Génération wordcloud pour un topic
model.generate_topic_wordcloud(1)

In [ ]:
# Extract all topics
topic_words, word_scores, topic_nums = model.get_topics()

for topic_id, words in zip(topic_nums, topic_words):
    print(f"Topic {topic_id}:")
    print(" ".join(words[:10]))  # top 10 words
    print()


In [ ]:
# Évaluer la diversité des topics
def topic_diversity(topic_words):
    """
    Calculate topic diversity: proportion of unique words among top words of all topics.
    """
    top_words = []
    for words in topic_words:   # just iterate directly
        top_words.extend(words)
    unique_words = set(top_words)
    diversity = len(unique_words) / len(top_words)
    return diversity

# Get topics from Top2Vec
topic_words, word_scores, topic_nums = model.get_topics()

# Calculate diversity
diversity_score = topic_diversity(topic_words)

print(f"Diversité des topics: {diversity_score:.2f}")  # closer to 1 = good diversity

In [14]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

def top2vec_coherence(model, docs, top_n_words=10, per_topic=False):
    """
    Compute coherence for Top2Vec topics.

    Parameters:
    - model: Top2Vec model (any version)
    - docs: list of documents
    - top_n_words: how many top words per topic to consider
    - per_topic: if True, returns coherence per topic; else returns average

    Returns:
    - coherence_score (float) or dict of topic_id -> coherence
    """
    # 1. Tokenize docs (basic split; can replace with more advanced preprocessing)
    tokenized_docs = [doc.split() for doc in docs_clean]

    # 2. Create Gensim dictionary
    dictionary = Dictionary(tokenized_docs)

    # 3. Get topics from Top2Vec
    topic_words, word_scores, topic_nums = model.get_topics()

    # 4. Build list of top words per topic
    topics_for_coherence = [words[:top_n_words] for words in topic_words]

    # 5. Compute coherence
    coherence_model = CoherenceModel(
        topics=topics_for_coherence,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )

    if per_topic:
        # Compute coherence for each topic individually
        topic_coherences = {}
        for idx, words in enumerate(topics_for_coherence):
            cm = CoherenceModel(
                topics=[words],
                texts=tokenized_docs,
                dictionary=dictionary,
                coherence='c_v'
            )
            topic_coherences[topic_nums[idx]] = cm.get_coherence()
        return topic_coherences

    else:
        # Average coherence
        return coherence_model.get_coherence()

In [15]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

def top2vec_coherence(model, docs, top_n_words=10, per_topic=False):
    """
    Compute coherence for Top2Vec topics.

    Parameters:
    - model: Top2Vec model (any version)
    - docs: list of documents
    - top_n_words: how many top words per topic to consider
    - per_topic: if True, returns coherence per topic; else returns average

    Returns:
    - coherence_score (float) or dict of topic_id -> coherence
    """
    # 1. Tokenize docs (basic split; can replace with more advanced preprocessing)
    tokenized_docs = [doc.split() for doc in docs_clean]

    # 2. Create Gensim dictionary
    dictionary = Dictionary(tokenized_docs)

    # 3. Get topics from Top2Vec
    topic_words, word_scores, topic_nums = model.get_topics()

    # 4. Build list of top words per topic
    topics_for_coherence = [words[:top_n_words] for words in topic_words]

    # 5. Compute coherence
    coherence_model = CoherenceModel(
        topics=topics_for_coherence,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='u_mass'
    )

    if per_topic:
        # Compute coherence for each topic individually
        topic_coherences = {}
        for idx, words in enumerate(topics_for_coherence):
            cm = CoherenceModel(
                topics=[words],
                texts=tokenized_docs,
                dictionary=dictionary,
                coherence='u_mass'
            )
            topic_coherences[topic_nums[idx]] = cm.get_coherence()
        return topic_coherences

    else:
        # Average coherence
        return coherence_model.get_coherence()

In [ ]:
score = top2vec_coherence(model, docs)
print(f"Average topic coherence: {score:.4f}")